<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 2</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 2: The skill registry</h1>
<p style="margin:.55em 0">The model can only <i>propose</i> a call: a name and JSON arguments. Our code decides what happens. The registry makes
any typed, documented function a tool:</p>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0"><b>schema from type hints</b> (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">str</code>, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">int</code>, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">float</code>, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">bool</code>,
      <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">Literal[...]</code> as an enum; a default makes an argument optional);</li>
  <li style="margin:.2em 0"><b>texts from the docstring</b>: first paragraph = description, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">When:</code> = one prompt line,
      <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">Args:</code> = argument help;</li>
  <li style="margin:.2em 0"><b>discovery</b>: every module in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/skills/</code> is imported; its file name is its <i>toolset</i>;</li>
  <li style="margin:.2em 0"><b>one guarded door</b>: bound size → parse → validate (allowlist) → guard → run with a timeout. Failures become data.</li>
</ul>

</div>

In [1]:
%%writefile build/app/registry.py
"""Skill registry: plain typed Python functions the model may call. Adding one = writing it in any module under
app/skills/. discover() imports those modules; the tool schema comes from the signature, the description,
the "When:" line and the per-argument help from the docstring. The module name is the skill's toolset;
agents choose toolsets in config.

    @skill(needs_patient=True)
    def cancel_appointment(ctx, appointment_id: int) -> dict:
        '''Cancel one of the verified patient's appointments.

        When: after the caller said yes to cancelling a specific appointment.
        Args:
            appointment_id: from my_appointments
        '''

Supported argument types: str, int, float, bool, typing.Literal[...] (an enum). An argument with a default
is optional. The first parameter is always `ctx`: the conversation (app.session.Session).

The model only PROPOSES a call. call() decides: bound the size, parse, validate against the schema
(allowlist), check the guard, run with a timeout. Every failure becomes {"error": ...}, never a crash.
"""
import asyncio
import importlib
import inspect
import json
import logging
import pkgutil
import re
import typing
from dataclasses import dataclass

log = logging.getLogger(__name__)

JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean"}
PY_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool}
MAX_ARGS_CHARS = 10_000  # bound size before parsing (trust boundary)


Writing build/app/registry.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">A <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">Skill</code> is everything known about one function, plus the exact tool definition the API expects.</p>

</div>

In [2]:
%%writefile -a build/app/registry.py

@dataclass(frozen=True)
class Skill:
    name: str
    toolset: str          # the module it lives in
    description: str      # what it does (sent to the model)
    when: str             # one line for the prompt: when to use it
    parameters: dict      # JSON schema of the arguments
    fn: typing.Callable
    needs_patient: bool   # guard: refuse unless the conversation has a verified patient

    @property
    def spec(self) -> dict:
        """The function-tool definition the Realtime API expects."""
        return {"type": "function", "name": self.name, "description": self.description, "parameters": self.parameters}


REGISTRY: dict[str, Skill] = {}


class SkillDefinitionError(TypeError):
    pass


Appending to build/app/registry.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The decorator works bare (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">@skill</code>) and with options (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">@skill(needs_patient=True)</code>). Two different
functions with one name is an error; the same function imported twice (a notebook re-run) just replaces itself.</p>

</div>

In [3]:
%%writefile -a build/app/registry.py

def skill(fn=None, *, needs_patient: bool = False, name: str | None = None):
    """Register a function as a skill. Usable bare (@skill) or with options (@skill(needs_patient=True))."""
    def register(f):
        s = describe(f, name or f.__name__, needs_patient)
        old = REGISTRY.get(s.name)
        if old and (old.fn.__module__, old.fn.__qualname__) != (f.__module__, f.__qualname__):
            raise SkillDefinitionError(f"skill '{s.name}' is defined twice: {old.fn.__module__} and {f.__module__}")
        REGISTRY[s.name] = s  # same function re-imported (e.g. a notebook re-run) simply replaces itself
        return f
    return register(fn) if fn else register


Appending to build/app/registry.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Reading a function: type hints become JSON schema; the docstring gives the texts. Every mistake a skill author can make is caught here, at import time.</p>

</div>

In [4]:
%%writefile -a build/app/registry.py

def _schema(t, where) -> dict:
    if typing.get_origin(t) is typing.Literal:
        values = typing.get_args(t)
        kinds = {type(v) for v in values}
        if len(kinds) != 1 or next(iter(kinds)) not in JSON_TYPES:
            raise SkillDefinitionError(f"{where}: Literal values must share one simple type")
        return {"type": JSON_TYPES[kinds.pop()], "enum": list(values)}
    if t in JSON_TYPES:
        return {"type": JSON_TYPES[t]}
    raise SkillDefinitionError(f"{where}: unsupported type {t!r} (use str, int, float, bool or Literal)")


def describe(f, name: str, needs_patient: bool = False) -> Skill:
    """Read a function into a Skill: schema from type hints, texts from the docstring."""
    doc = inspect.getdoc(f)
    if not doc:
        raise SkillDefinitionError(f"{name}: needs a docstring (the model reads it)")
    description = doc.split("\n\n")[0].replace("\n", " ").strip()
    when = (m[1].strip() if (m := re.search(r"^When:\s*(.+)$", doc, re.M)) else "")
    args_help = {}
    if m := re.search(r"^Args:\s*\n((?:[ \t]+.+\n?)+)", doc, re.M):
        for line in m[1].splitlines():
            if am := re.match(r"\s+(\w+):\s*(.+)", line):
                args_help[am[1]] = am[2].strip()
    params = list(inspect.signature(f).parameters.values())
    if not params or params[0].name != "ctx":
        raise SkillDefinitionError(f"{name}: the first parameter must be `ctx`")
    hints = typing.get_type_hints(f)
    props, required = {}, []
    for p in params[1:]:
        if p.name not in hints:
            raise SkillDefinitionError(f"{name}.{p.name}: needs a type hint")
        props[p.name] = _schema(hints[p.name], f"{name}.{p.name}")
        if p.name in args_help:
            props[p.name]["description"] = args_help[p.name]
        if p.default is inspect.Parameter.empty:
            required.append(p.name)
    unknown_help = set(args_help) - set(props)
    if unknown_help:
        raise SkillDefinitionError(f"{name}: docstring describes unknown argument(s) {sorted(unknown_help)}")
    parameters = {"type": "object", "properties": props, "required": required, "additionalProperties": False}
    return Skill(name, f.__module__.rsplit(".", 1)[-1], description, when, parameters, f, needs_patient)


Appending to build/app/registry.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Discovery and selection: import every skill module, then give each agent only its toolsets.</p>

</div>

In [5]:
%%writefile -a build/app/registry.py

def discover(package: str = "app.skills") -> dict[str, list[str]]:
    """Import every module of `package` (their @skill decorators register them). Returns {toolset: [skills]}."""
    pkg = importlib.import_module(package)
    for m in sorted(pkgutil.iter_modules(pkg.__path__), key=lambda m: m.name):
        if not m.name.startswith("_"):
            importlib.import_module(f"{package}.{m.name}")
    sets: dict[str, list[str]] = {}
    for s in REGISTRY.values():
        sets.setdefault(s.toolset, []).append(s.name)
    return sets


def select(toolsets) -> dict[str, Skill]:
    """The skills of some toolsets ("*" = all), in a stable order."""
    known = {s.toolset for s in REGISTRY.values()}
    if missing := [t for t in toolsets if t != "*" and t not in known]:
        raise SkillDefinitionError(f"unknown toolset(s) {missing}; known: {sorted(known)}")
    return {s.name: s for s in sorted(REGISTRY.values(), key=lambda s: (s.toolset, s.name))
            if "*" in toolsets or s.toolset in toolsets}


Appending to build/app/registry.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">validate</code> is an allowlist: unknown arguments, missing ones, wrong types and values outside an enum are
all refused. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">call</code> is the one door: sync skills run on a worker thread so a slow database can't freeze the audio.</p>

</div>

In [6]:
%%writefile -a build/app/registry.py

def validate(args, params: dict) -> str | None:
    """Allowlist check of parsed arguments against a flat JSON schema. Returns an error message or None."""
    if not isinstance(args, dict):
        return "arguments must be a JSON object"
    props = params["properties"]
    if extra := sorted(set(args) - set(props)):
        return f"unknown argument(s): {extra}"
    if missing := [k for k in params["required"] if k not in args]:
        return f"missing argument(s): {missing}"
    for k, v in args.items():
        t = props[k]["type"]
        if isinstance(v, bool) and t != "boolean":  # bool is an int subclass in Python
            return f"{k}: expected {t}, got boolean"
        if not isinstance(v, PY_TYPES[t]):
            return f"{k}: expected {t}, got {type(v).__name__}"
        if "enum" in props[k] and v not in props[k]["enum"]:
            return f"{k}: must be one of {props[k]['enum']}"
    return None


async def call(s: Skill, ctx, arguments: str, timeout: float) -> dict:
    """Run one proposed call. Never raises: every failure is returned as {"error": ...} (and logged)."""
    if len(arguments) > MAX_ARGS_CHARS:
        return {"error": "arguments too large"}
    try:
        args = json.loads(arguments or "{}")
    except json.JSONDecodeError as e:
        return {"error": f"arguments are not valid JSON: {e}"}
    if err := validate(args, s.parameters):
        return {"error": err}
    if s.needs_patient and not getattr(ctx, "patient", None):
        return {"status": "needs_verification", "problem": "the caller is not verified yet",
                "next": "Ask for their mobile number and call verify_patient."}
    try:
        if inspect.iscoroutinefunction(s.fn):
            out = await asyncio.wait_for(s.fn(ctx, **args), timeout)
        else:  # sync skills run on a worker thread, so a slow one can't freeze the audio loop
            out = await asyncio.wait_for(asyncio.to_thread(s.fn, ctx, **args), timeout)
    except asyncio.TimeoutError:
        log.error("skill %s timed out after %ss", s.name, timeout)
        return {"error": f"{s.name} took too long; tell the caller and offer to try again"}
    except Exception as e:
        log.exception("skill %s failed", s.name)
        return {"error": f"{type(e).__name__}: {e}"}
    if not isinstance(out, dict):
        log.error("skill %s returned %s, not a dict", s.name, type(out).__name__)
        return {"error": f"{s.name} returned an invalid result"}
    return out


Appending to build/app/registry.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it</h2><p style="margin:.55em 0">A skill defined right here:</p>

</div>

In [7]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [8]:
from typing import Literal
from app import registry

@registry.skill
def convert(ctx, value: float, to: Literal['C', 'F'], digits: int = 1) -> dict:
    """Convert a temperature.

    When: when asked to convert a temperature.
    Args:
        digits: decimals in the answer
    """
    r = value * 9 / 5 + 32 if to == 'F' else (value - 32) * 5 / 9
    return {'result': round(r, digits)}

registry.REGISTRY['convert'].spec

{'type': 'function',
 'name': 'convert',
 'description': 'Convert a temperature.',
 'parameters': {'type': 'object',
  'properties': {'value': {'type': 'number'},
   'to': {'type': 'string', 'enum': ['C', 'F']},
   'digits': {'type': 'integer', 'description': 'decimals in the answer'}},
  'required': ['value', 'to'],
  'additionalProperties': False}}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The model's proposals go through <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">call</code>. Good ones run; bad ones become error data the model can explain:</p>

</div>

In [9]:
sk = registry.REGISTRY['convert']
for args in ('{"value": 30, "to": "F"}', '{"value": 30, "to": "K"}', '{"value": "30", "to": "F"}',
             '{"value": 30, "to": "F", "admin": true}', '{"value": 30'):
    print(args, '->', await registry.call(sk, None, args, timeout=2))

{"value": 30, "to": "F"} -> {'result': 86.0}
{"value": 30, "to": "K"} -> {'error': "to: must be one of ['C', 'F']"}
{"value": "30", "to": "F"} -> {'error': 'value: expected number, got str'}
{"value": 30, "to": "F", "admin": true} -> {'error': "unknown argument(s): ['admin']"}
{"value": 30 -> {'error': "arguments are not valid JSON: Expecting ',' delimiter: line 1 column 13 (char 12)"}


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Guards and timeouts are enforced by the registry, not trusted to the prompt:</p>

</div>

In [10]:
import time

class Ctx:
    patient = None

@registry.skill(needs_patient=True)
def secret(ctx) -> dict:
    """Only for verified callers."""
    return {'ok': True}

@registry.skill
def slow(ctx) -> dict:
    """Takes too long."""
    time.sleep(1.5)
    return {}

await registry.call(registry.REGISTRY['secret'], Ctx(), '{}', 2), await registry.call(registry.REGISTRY['slow'], Ctx(), '{}', 0.3)

skill slow timed out after 0.3s


({'status': 'needs_verification',
  'problem': 'the caller is not verified yet',
  'next': 'Ask for their mobile number and call verify_patient.'},
 {'error': 'slow took too long; tell the caller and offer to try again'})

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Authoring mistakes fail at import time with a clear message:</p>

</div>

In [11]:
def no_doc(ctx, x: int):
    return {}

def no_hint(ctx, x):
    """Doc."""

for f in (no_doc, no_hint):
    try:
        registry.describe(f, f.__name__)
    except registry.SkillDefinitionError as e:
        print('refused:', e)

refused: no_doc: needs a docstring (the model reads it)
refused: no_hint.x: needs a type hint


In [12]:
for n in ('convert', 'secret', 'slow'):
    registry.REGISTRY.pop(n)   # demo skills out of the registry

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">03_storage_and_rules.ipynb</code>: the clinic's data and its rules.</p>

</div>